# 🌸 Project 2: Data Classification 
### DecodeLabs | Industrial Training Kit — Batch 2026

---

**Goal:** Build a basic classification model using the Iris dataset.  
**Algorithm:** K-Nearest Neighbors (KNN)  
**Pipeline:** Load → Scale → Split → Train → Evaluate  

---

## Step 1: Import Libraries

In [ ]:
# Core Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn — the ML toolkit
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score
)

import warnings
warnings.filterwarnings('ignore')

print("✅ All libraries imported successfully!")

## Step 2: Load & Explore the Dataset

> **The Iris Benchmark** — 150 balanced samples, 3 classes (Setosa, Versicolor, Virginica), 4 features (Sepal Length, Sepal Width, Petal Length, Petal Width).
>
> Dataset source: [Iris Dataset on Kaggle](https://www.kaggle.com/datasets/uciml/iris)

In [ ]:
# Load directly from sklearn (same as Kaggle UCI Iris dataset)
iris = load_iris()

# Build a clean DataFrame
df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
df['species'] = pd.Categorical.from_codes(iris.target, iris.target_names)
df['target'] = iris.target

print(f"Dataset Shape: {df.shape}")
print(f"Classes: {list(iris.target_names)}")
print(f"Features: {list(iris.feature_names)}")
print()
df.head(10)

In [ ]:
# Basic statistics
print("📊 Dataset Statistics:")
df.describe()

In [ ]:
# Check for missing values
print("🔍 Missing Values:")
print(df.isnull().sum())
print()
print("📦 Class Distribution (Balanced?):")
print(df['species'].value_counts())

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Pairplot — visualize all feature relationships
sns.set_style("whitegrid")
sns.pairplot(df, hue='species', palette='Set2', diag_kind='kde', height=2.5)
plt.suptitle('Iris Dataset — Pairwise Feature Relationships', y=1.02, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df[iris.feature_names].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, square=True)
plt.title('Feature Correlation Heatmap', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Box plots per feature
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for i, feature in enumerate(iris.feature_names):
    sns.boxplot(data=df, x='species', y=feature, palette='Set2', ax=axes[i])
    axes[i].set_title(f'{feature}', fontweight='bold')
    axes[i].set_xlabel('')

plt.suptitle('Feature Distribution by Species', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## Step 4: Feature Scaling — The Gatekeeper Rule

> KNN relies on **distance** between points. Without scaling, features with large ranges (like petal length in cm) will dominate over small-range features. We use `StandardScaler` to bring everything to **mean=0, variance=1**.

In [ ]:
# Define features (X) and target (y)
X = df[iris.feature_names].values
y = df['target'].values

# Apply StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Quick visual proof of scaling
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].scatter(X[:, 0], X[:, 2], c=y, cmap='Set2', edgecolors='k', alpha=0.7)
axes[0].set_title('RAW DATA (Biased)', fontweight='bold')
axes[0].set_xlabel('Sepal Length (cm)')
axes[0].set_ylabel('Petal Length (cm)')

axes[1].scatter(X_scaled[:, 0], X_scaled[:, 2], c=y, cmap='Set2', edgecolors='k', alpha=0.7)
axes[1].set_title('STANDARD SCALED (Balanced)', fontweight='bold')
axes[1].set_xlabel('Sepal Length (scaled)')
axes[1].set_ylabel('Petal Length (scaled)')

plt.suptitle('Effect of StandardScaler on Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Before scaling — Mean: {X[:, 0].mean():.2f}, Std: {X[:, 0].std():.2f}")
print(f"After  scaling — Mean: {X_scaled[:, 0].mean():.2f}, Std: {X_scaled[:, 0].std():.2f}")

## Step 5: Train-Test Split — Structural Integrity

> We use an **80/20 split** with shuffling enabled (`random_state=42` for reproducibility).  
> The test set is **locked** — the model never sees it during training.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,       # 20% for testing
    random_state=42,      # reproducibility
    shuffle=True,         # remove order bias
    stratify=y            # keep class balance in both sets
)

print(f"✅ Training set size : {X_train.shape[0]} samples")
print(f"✅ Testing  set size : {X_test.shape[0]} samples")
print()
print("Training class distribution:", dict(zip(*np.unique(y_train, return_counts=True))))
print("Testing  class distribution:", dict(zip(*np.unique(y_test,  return_counts=True))))

## Step 6: Tuning the Engine — Finding the Optimal K

> **The Elbow Method:** Test different K values and pick the one at the "elbow" — lowest error rate without overfitting (K=1) or underfitting (K=large).

In [ ]:
error_rates = []
k_range = range(1, 31)

for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, y_train)
    preds = knn.predict(X_test)
    error_rates.append(1 - accuracy_score(y_test, preds))

best_k = k_range[np.argmin(error_rates)]

plt.figure(figsize=(10, 5))
plt.plot(k_range, error_rates, color='steelblue', linestyle='--',
         marker='o', markerfacecolor='white', markersize=8, linewidth=2)
plt.scatter([best_k], [min(error_rates)], color='orangered', s=200,
            zorder=5, label=f'Optimal K = {best_k} (The Elbow)')
plt.title('Tuning the Engine: Choosing K', fontsize=14, fontweight='bold')
plt.xlabel('K Value')
plt.ylabel('Error Rate')
plt.legend()
plt.tight_layout()
plt.show()

print(f"🎯 Optimal K = {best_k} with Error Rate = {min(error_rates):.4f}")

## Step 7: Train the KNN Model

> **The Scikit-learn Workflow:**  
> 1. **INSTANTIATE** — Build the frame  
> 2. **FIT** — Memorize the map  
> 3. **PREDICT** — Apply logic

In [ ]:
# Step 1: Instantiate
model = KNeighborsClassifier(n_neighbors=best_k)

# Step 2: Fit (train)
model.fit(X_train, y_train)

# Step 3: Predict
y_pred = model.predict(X_test)

print(f"✅ Model trained with K = {best_k}")
print()
print("Sample Predictions vs Actual:")
comparison = pd.DataFrame({
    'Actual'   : [iris.target_names[i] for i in y_test[:10]],
    'Predicted': [iris.target_names[i] for i in y_pred[:10]],
    'Correct'  : ['✅' if a == p else '❌' for a, p in zip(y_test[:10], y_pred[:10])]
})
print(comparison.to_string(index=False))

## Step 8: Output Validation — Confusion Matrix & Metrics

> **Accuracy alone is a mirage** on imbalanced data. We validate with:  
> - **Confusion Matrix** — TP, FP, FN, TN breakdown  
> - **F1 Score** — Harmonic mean of Precision & Recall  
> - **Classification Report** — Full per-class analysis

In [ ]:
# ── Metrics ──
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print("=" * 45)
print(f"  Accuracy : {accuracy * 100:.2f}%")
print(f"  F1 Score : {f1:.4f}")
print("=" * 45)
print()
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

In [ ]:
# ── Confusion Matrix ──
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names,
            linewidths=1, linecolor='white',
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('The Diagnostic Tool: Confusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('Actual Label', fontweight='bold')
plt.xlabel('Predicted Label', fontweight='bold')
plt.tight_layout()
plt.show()

print("TP (diagonal) = Correct predictions")
print("FP / FN (off-diagonal) = Misclassifications")

## Step 9: Decision Boundary Visualization

> Plotting the KNN decision boundary using the 2 most discriminative features (Petal Length & Petal Width).

In [ ]:
# Use petal length & petal width (features 2 & 3) for 2D boundary
X_2d = X_scaled[:, 2:4]
X_train_2d, X_test_2d, y_train_2d, y_test_2d = train_test_split(
    X_2d, y, test_size=0.2, random_state=42, stratify=y)

knn_2d = KNeighborsClassifier(n_neighbors=best_k)
knn_2d.fit(X_train_2d, y_train_2d)

# Create mesh grid
h = 0.02
x_min, x_max = X_2d[:, 0].min() - 1, X_2d[:, 0].max() + 1
y_min, y_max = X_2d[:, 1].min() - 1, X_2d[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = knn_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 6))
plt.contourf(xx, yy, Z, alpha=0.3, cmap='Set2')
colors = ['#4CAF50', '#2196F3', '#FF5722']
for i, cls in enumerate(iris.target_names):
    mask = y_test_2d == i
    plt.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
                color=colors[i], label=cls, edgecolors='k',
                s=80, zorder=3)

plt.title(f'KNN Decision Boundary (K={best_k}) — Petal Features', fontsize=13, fontweight='bold')
plt.xlabel('Petal Length (scaled)')
plt.ylabel('Petal Width (scaled)')
plt.legend(title='Species')
plt.tight_layout()
plt.show()

## Step 10: Predict on New / Unseen Data

In [ ]:
# Example: Predict species for a new flower measurement
# [sepal_length, sepal_width, petal_length, petal_width] in cm
new_samples = np.array([
    [5.1, 3.5, 1.4, 0.2],   # looks like Setosa
    [6.7, 3.0, 5.2, 2.3],   # looks like Virginica
    [5.5, 2.6, 4.4, 1.2],   # looks like Versicolor
])

# Scale using the SAME scaler fitted on training data
new_samples_scaled = scaler.transform(new_samples)

predictions = model.predict(new_samples_scaled)
probabilities = model.predict_proba(new_samples_scaled)

print("🌸 New Sample Predictions:")
print("-" * 60)
for i, (pred, prob) in enumerate(zip(predictions, probabilities)):
    species = iris.target_names[pred]
    confidence = max(prob) * 100
    print(f"Sample {i+1}: {new_samples[i].tolist()}")
    print(f"  → Predicted: {species.upper()} (Confidence: {confidence:.1f}%)")
    print()

## 📊 Final Summary — Project 2 Complete!

---

| Step | Description | Status |
|------|-------------|--------|
| 1 | Import Libraries | ✅ |
| 2 | Load & Explore Iris Dataset | ✅ |
| 3 | Exploratory Data Analysis | ✅ |
| 4 | Feature Scaling (StandardScaler) | ✅ |
| 5 | Train-Test Split (80/20) | ✅ |
| 6 | Tune K using Elbow Method | ✅ |
| 7 | Train KNN Classifier | ✅ |
| 8 | Confusion Matrix + F1 Score | ✅ |
| 9 | Decision Boundary Visualization | ✅ |
| 10 | Predict on New Data | ✅ |

---

### 🔑 Key Takeaways

- **KNN** classifies by majority vote among the K nearest training points — no explicit training, just memorization.
- **Feature Scaling** is mandatory for distance-based algorithms like KNN.
- **Accuracy alone is misleading** — always use F1 Score and Confusion Matrix.
- **The Elbow Method** helps select the optimal K to avoid overfitting (K=1) and underfitting (large K).

---
*DecodeLabs | Batch 2026 — Project 2 Submitted ✅*